In [3]:
from pathlib import Path

# I locate the project root relative to my notebook directory.
PROJECT_ROOT = Path.cwd().resolve().parent

# I define reusable project directories.
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

# I create the required directories if they don't exist.
for directory in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    MODELS_DIR,
    OUTPUTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# I verify that Jupyter is running from the expected location.
assert (PROJECT_ROOT / "notebooks").is_dir(), (
    "Project root not found. Launch this notebook from the notebooks directory."
)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Processed data:", PROCESSED_DATA_DIR)
print("Models:", MODELS_DIR)
print("Outputs:", OUTPUTS_DIR)

Project root: C:\Users\JONATHAN\Desktop\WORK\spam-phishing-detector
Raw data: C:\Users\JONATHAN\Desktop\WORK\spam-phishing-detector\data\raw
Processed data: C:\Users\JONATHAN\Desktop\WORK\spam-phishing-detector\data\processed
Models: C:\Users\JONATHAN\Desktop\WORK\spam-phishing-detector\models
Outputs: C:\Users\JONATHAN\Desktop\WORK\spam-phishing-detector\outputs


In [5]:
# I define the expected dataset locations.
SMS_PATH = RAW_DATA_DIR / "SMSSpamCollection"
PHISHING_PATH = RAW_DATA_DIR / "meajor_cleaned_preprocessed.parquet.gzip"

# I check that both datasets are available.
datasets = {
    "SMS Spam Collection": SMS_PATH,
    "MeAJOR Phishing Emails": PHISHING_PATH,
}

for name, path in datasets.items():
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{name}: FOUND ({size_mb:.2f} MB)")
    else:
        print(f"{name}: NOT FOUND")

SMS Spam Collection: FOUND (0.46 MB)
MeAJOR Phishing Emails: FOUND (82.28 MB)


## Load and inspect both datasets

In [7]:
import pandas as pd

# I load the SMS dataset using its original tab-separated format.
sms_df = pd.read_csv(
    SMS_PATH,
    sep="\t",
    header=None,
    names=["label", "text"],
    encoding="utf-8"
)

# I load the compressed Parquet email dataset.
phishing_df = pd.read_parquet(PHISHING_PATH)

print("SMS dataset shape:", sms_df.shape)
print("Phishing dataset shape:", phishing_df.shape)

SMS dataset shape: (5572, 2)
Phishing dataset shape: (108685, 20)


In [10]:
# I inspect the original columns and their data types.
print("SMS DATASET")
print(sms_df.info())

print("\nPHISHING EMAIL DATASET")
print(phishing_df.info())

SMS DATASET
<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   label   5572 non-null   str  
 1   text    5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB
None

PHISHING EMAIL DATASET
<class 'pandas.DataFrame'>
RangeIndex: 108685 entries, 0 to 108684
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   sender            108685 non-null  str    
 1   sender_domain     108191 non-null  str    
 2   receiver          108685 non-null  str    
 3   receiver_domain   106314 non-null  str    
 4   date              108685 non-null  str    
 5   subject           107230 non-null  str    
 6   content_types     108684 non-null  str    
 7   body              108684 non-null  str    
 8   urls              64052 non-null   str    
 9   url_count         108685 non-null  float64
 10  url_le

In [12]:
# I inspect the class distributions before preprocessing.
print("SMS LABEL DISTRIBUTION")
print(sms_df["label"].value_counts(dropna=False))

print("\nPHISHING EMAIL LABEL DISTRIBUTION")

if "label" in phishing_df.columns:
    print(phishing_df["label"].value_counts(dropna=False))
else:
    print("Label column not found. Available columns:")
    print(phishing_df.columns.tolist())

# I inspect a few SMS examples.
display(sms_df.head(3))

# I inspect the email dataset structure without printing
# potentially sensitive or excessively long message bodies.
display(phishing_df.head(3))

SMS LABEL DISTRIBUTION
label
ham     4825
spam     747
Name: count, dtype: int64

PHISHING EMAIL LABEL DISTRIBUTION
label
0.0    60650
1.0    48034
NaN        1
Name: count, dtype: int64


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...


,sender,sender_domain,receiver,receiver_domain,date,subject,content_types,body,urls,url_count,url_length_max,url_length_avg,url_subdom_max,url_subdom_avg,attachment_count,has_attachments,attachment_types,language,source,label
0,d66e9e64b006d6bca649f1c945129c42c43836872b2ead...,enron.com,35c5a9fb9fba3b8737ed7cef2a87e427a73db4fca85f6b...,enron.com,2001-06-29 09:37:04-05:00,[ORGANIZATION] failover plan.,text/plain,"Hi [NAME], \n\nTonight we are rolling out a n...",NaN,0.0,0.0,0.0,0.0,0.0,0.0,False,NaN,en,trec5,0.0
1,0907d5c64598aa2639154ed4e1556be615669e40052a1f...,enron.com,aa2c35499eae5999bf6080453cc719a891da2bb0c3803d...,enron.com,2001-06-29 08:39:30-05:00,RE: Intranet Site,text/plain,"[NAME] r these new?\tIntranet Site\n\n[NAME],\...",http://eastpower.dev.corp.enron.com/summary/pj...,3.0,60.0,58.0,3.0,3.0,0.0,False,NaN,en,trec5,0.0
2,7c3201a5ff8c5985218f1e3f11e330dc0242bbd28c6c20...,enron.com,a736837579feb601fbf6c0657d3d93689774afa6491bb9...,enron.com;enron.com,2001-06-29 10:35:17-05:00,FW: [ORGANIZATION] Company information,text/plain,"[NAME]/[NAME],\n\nWe are currently trading und...",NaN,0.0,0.0,0.0,0.0,0.0,0.0,False,NaN,en,trec5,0.0


## Data Quality Assessment

### Missing values and duplicates

I examine missing values and duplicate records before preparing the datasets for modeling. This helps me identify records that may require cleaning and potential duplication that could inflate evaluation performance.

In [20]:
# I summarize missing values in both datasets.
print("SMS DATASET — MISSING VALUES")
print(sms_df.isna().sum())

print("\nPHISHING DATASET — MISSING VALUES")
missing = phishing_df.isna().sum()

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": (missing / len(phishing_df) * 100).round(2)
})

display(
    missing_summary.sort_values(
        "missing_count",
        ascending=False
    )
)

# I check exact duplicate rows.
print("\nEXACT DUPLICATE ROWS")
print("SMS:", sms_df.duplicated().sum())
print("Phishing:", phishing_df.duplicated().sum())

# I check duplicate message content.
print("\nDUPLICATE MESSAGE CONTENT")

print(
    "SMS:",
    sms_df.duplicated(subset=["text"]).sum()
)

print(
    "Phishing:",
    phishing_df.duplicated(
        subset=["subject", "body"]
    ).sum()
)

SMS DATASET — MISSING VALUES
label    0
text     0
dtype: int64

PHISHING DATASET — MISSING VALUES


,missing_count,missing_pct
attachment_types,107019,98.47
urls,44633,41.07
receiver_domain,2371,2.18
subject,1455,1.34
sender_domain,494,0.45
language,1,0.00
source,1,0.00
body,1,0.00
label,1,0.00
content_types,1,0.00



EXACT DUPLICATE ROWS
SMS: 403
Phishing: 13

DUPLICATE MESSAGE CONTENT
SMS: 403
Phishing: 3751


## Examine phishing dataset sources

In [23]:
# I examine the distribution of phishing labels
# across the original dataset sources.

source_distribution = pd.crosstab(
    phishing_df["source"],
    phishing_df["label"],
    margins=True
)

display(source_distribution)

label,0.0,1.0,All
source,,,
trec5,29929,19654,49583
trec6,11208,3797,15005
trec7,19513,24583,44096
All,60650,48034,108684


## Check for Conflicting Labels

### Check label consistency
I check whether identical message content has conflicting classifications before removing duplicates. This prevents me from arbitrarily retaining one label when the same message has been assigned different labels.

In [27]:
# I check whether identical SMS messages have conflicting labels.
sms_conflicts = (
    sms_df.groupby("text")["label"]
    .nunique()
)

print("SMS messages with conflicting labels:")
print((sms_conflicts > 1).sum())


# I check whether identical email subjects and bodies
# have conflicting phishing labels.
email_conflicts = (
    phishing_df.dropna(subset=["label"])
    .groupby(
        ["subject", "body"],
        dropna=False
    )["label"]
    .nunique()
)

print("\nEmail messages with conflicting labels:")
print((email_conflicts > 1).sum())


# I examine the number of distinct message groups.
print("\nUNIQUE MESSAGE GROUPS")
print("SMS:", sms_df["text"].nunique())

print(
    "Phishing:",
    phishing_df[["subject", "body"]]
    .drop_duplicates()
    .shape[0]
)

SMS messages with conflicting labels:
0

Email messages with conflicting labels:
0

UNIQUE MESSAGE GROUPS
SMS: 5169
Phishing: 104934


## Inspect Date Quality Before Cleaning
I examine the email timestamps before deciding how to split the phishing dataset. My goal is to evaluate whether chronological splitting is appropriate and avoid introducing temporal leakage into model evaluation.

In [31]:
# I convert the email timestamps into a consistent UTC format.
email_dates = pd.to_datetime(
    phishing_df["date"],
    errors="coerce",
    utc=True
)

print("EMAIL DATE QUALITY")
print("Total records:", len(phishing_df))
print("Valid dates:", email_dates.notna().sum())
print("Invalid dates:", email_dates.isna().sum())

print("\nDATE RANGE")
print("Earliest:", email_dates.min())
print("Latest:", email_dates.max())

# I inspect the distribution of messages across years.
print("\nRECORDS BY YEAR")
print(
    email_dates.dt.year
    .value_counts(dropna=False)
    .sort_index()
)

# I inspect date coverage across the original data sources.
date_summary = (
    phishing_df.assign(parsed_date=email_dates)
    .groupby("source", dropna=False)
    .agg(
        records=("label", "size"),
        earliest_date=("parsed_date", "min"),
        latest_date=("parsed_date", "max"),
        invalid_dates=("parsed_date", lambda x: x.isna().sum())
    )
)

print("\nDATE COVERAGE BY SOURCE")
display(date_summary)

EMAIL DATE QUALITY
Total records: 108685
Valid dates: 108685
Invalid dates: 0

DATE RANGE
Earliest: 1889-05-19 08:59:51+00:00
Latest: 2102-03-16 15:16:56+00:00

RECORDS BY YEAR
date
1889     2
1890     1
1893     4
1894    16
1895     5
        ..
2080     1
2087     2
2090     4
2099     1
2102     2
Name: count, Length: 71, dtype: int64

DATE COVERAGE BY SOURCE


,records,earliest_date,latest_date,invalid_dates
source,,,,
trec5,49583,1898-06-16 08:01:06+00:00,2034-11-21 06:42:28+00:00,0
trec6,15005,1889-05-19 08:59:51+00:00,2019-10-29 21:53:50+00:00,0
trec7,44096,1900-06-27 19:57:30+00:00,2102-03-16 15:16:56+00:00,0
NaN,1,1994-09-02 01:19:11+00:00,1994-09-02 01:19:11+00:00,0


## Cleaning The Datasets
I prepare the SMS and phishing datasets for modeling by removing duplicate message content, handling missing values, and standardizing labels.

I preserve the original datasets and save cleaned copies separately. I also exclude fields that could introduce dataset-specific artifacts into the initial text models.

Because the phishing email timestamps contain implausible dates, I will not use them for chronological splitting or as predictive features.

### Cleaning  the SMS dataset

In [39]:
# I create a copy to preserve the original SMS dataset.
sms_clean = sms_df.copy()

# I remove exact duplicate messages.
sms_clean = sms_clean.drop_duplicates(
    subset=["text"]
).copy()

# I standardize the binary labels.
sms_clean["label"] = sms_clean["label"].map({
    "ham": 0,
    "spam": 1
})

# I verify that all labels are valid.
assert sms_clean["label"].notna().all()

sms_clean["label"] = sms_clean["label"].astype("int8")

# I reset the index after deduplication.
sms_clean = sms_clean.reset_index(drop=True)

print("CLEANED SMS DATASET")
print("Shape:", sms_clean.shape)
print("\nLabel distribution:")
print(sms_clean["label"].value_counts())

# I save the cleaned dataset using a portable path.
sms_clean.to_csv(
    PROCESSED_DATA_DIR / "sms_clean.csv",
    index=False
)

CLEANED SMS DATASET
Shape: (5169, 2)

Label distribution:
label
0    4516
1     653
Name: count, dtype: int64


## Cleaning the phishing dataset

In [42]:
# I select the fields needed for text modeling and evaluation.
phishing_clean = phishing_df[
    ["subject", "body", "label", "source", "date"]
].copy()

# I remove records without a valid classification.
phishing_clean = phishing_clean[
    phishing_clean["label"].isin([0, 1])
].copy()

# I replace missing subject and body values with empty strings.
phishing_clean["subject"] = (
    phishing_clean["subject"].fillna("")
)

phishing_clean["body"] = (
    phishing_clean["body"].fillna("")
)

# I remove duplicate email content.
phishing_clean = phishing_clean.drop_duplicates(
    subset=["subject", "body"]
).copy()

# I standardize the labels.
phishing_clean["label"] = (
    phishing_clean["label"].astype("int8")
)

# I create the text field for the initial phishing model.
phishing_clean["text"] = (
    phishing_clean["subject"].str.strip()
    + "\n"
    + phishing_clean["body"].str.strip()
).str.strip()

# I remove messages with no usable text.
phishing_clean = phishing_clean[
    phishing_clean["text"].str.len() > 0
].copy()

phishing_clean = phishing_clean.reset_index(drop=True)

print("CLEANED PHISHING DATASET")
print("Shape:", phishing_clean.shape)

print("\nLabel distribution:")
print(phishing_clean["label"].value_counts())

print("\nSources:")
print(phishing_clean["source"].value_counts())

# I save the cleaned dataset.
phishing_clean.to_parquet(
    PROCESSED_DATA_DIR / "phishing_clean.parquet",
    index=False
)

CLEANED PHISHING DATASET
Shape: (104933, 6)

Label distribution:
label
0    58088
1    46845
Name: count, dtype: int64

Sources:
source
trec5    46853
trec7    43375
trec6    14705
Name: count, dtype: int64


## Basic cleaning validation

In [45]:
print("CLEANING VALIDATION")

print("\nSMS")
print("Missing labels:", sms_clean["label"].isna().sum())
print("Duplicate texts:", sms_clean["text"].duplicated().sum())
print("Empty texts:", sms_clean["text"].str.strip().eq("").sum())

print("\nPHISHING")
print("Missing labels:", phishing_clean["label"].isna().sum())

print(
    "Duplicate subject/body:",
    phishing_clean.duplicated(
        subset=["subject", "body"]
    ).sum()
)

print(
    "Empty texts:",
    phishing_clean["text"].str.strip().eq("").sum()
)

print("\nSAVED FILES")
print(PROCESSED_DATA_DIR / "sms_clean.csv")
print(PROCESSED_DATA_DIR / "phishing_clean.parquet")

CLEANING VALIDATION

SMS
Missing labels: 0
Duplicate texts: 0
Empty texts: 0

PHISHING
Missing labels: 0
Duplicate subject/body: 0
Empty texts: 0

SAVED FILES
C:\Users\JONATHAN\Desktop\WORK\spam-phishing-detector\data\processed\sms_clean.csv
C:\Users\JONATHAN\Desktop\WORK\spam-phishing-detector\data\processed\phishing_clean.parquet


## Detect normalized duplicates
I check for messages that become identical after lightweight text normalization. Although I already removed exact duplicates, minor formatting differences could still cause substantially identical messages to appear in separate evaluation partitions.

I use normalized fingerprints to identify these cases before constructing train, validation and test sets.

In [48]:
import re
import hashlib


def normalize_for_fingerprint(text: str) -> str:
    """Normalize text for duplicate detection, not model training."""
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()


def make_fingerprint(text: str) -> str:
    """Create a deterministic fingerprint of normalized text."""
    normalized = normalize_for_fingerprint(text)
    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()


# I compute normalized fingerprints without modifying model inputs.
sms_clean["fingerprint"] = (
    sms_clean["text"].map(make_fingerprint)
)

phishing_clean["fingerprint"] = (
    phishing_clean["text"].map(make_fingerprint)
)


# I summarize normalized duplicate records.
for name, df in [
    ("SMS", sms_clean),
    ("Phishing", phishing_clean)
]:
    duplicate_count = df["fingerprint"].duplicated().sum()

    conflicting_groups = (
        df.groupby("fingerprint")["label"]
        .nunique()
        .gt(1)
        .sum()
    )

    print(f"\n{name.upper()}")
    print("Records:", len(df))
    print("Normalized duplicates:", duplicate_count)
    print("Conflicting label groups:", conflicting_groups)


SMS
Records: 5169
Normalized duplicates: 39
Conflicting label groups: 0

PHISHING
Records: 104933
Normalized duplicates: 582
Conflicting label groups: 0


## Remove Normalized Duplicates
I remove messages that become identical after lightweight normalization. This reduces the risk of nearly identical examples appearing in both training and evaluation datasets.

I retain the original message text for modeling because the normalized fingerprints are intended only for duplicate detection, not as model features.

In [52]:
# I create separate modeling-ready datasets.
sms_final = (
    sms_clean
    .drop_duplicates(subset=["fingerprint"])
    .drop(columns=["fingerprint"])
    .reset_index(drop=True)
    .copy()
)

phishing_final = (
    phishing_clean
    .drop_duplicates(subset=["fingerprint"])
    .drop(columns=["fingerprint"])
    .reset_index(drop=True)
    .copy()
)

# I verify that the expected duplicate records were removed.
assert len(sms_final) == 5130
assert len(phishing_final) == 104351

# I ensure all remaining labels are valid.
assert sms_final["label"].isin([0, 1]).all()
assert phishing_final["label"].isin([0, 1]).all()

print("FINAL SMS DATASET")
print("Shape:", sms_final.shape)
print(sms_final["label"].value_counts())

print("\nFINAL PHISHING DATASET")
print("Shape:", phishing_final.shape)
print(phishing_final["label"].value_counts())

FINAL SMS DATASET
Shape: (5130, 2)
label
0    4499
1     631
Name: count, dtype: int64

FINAL PHISHING DATASET
Shape: (104351, 6)
label
0    57842
1    46509
Name: count, dtype: int64


#### Saving the modeling-ready datasets

In [56]:
# I save the final deduplicated datasets using portable paths.
sms_final.to_csv(
    PROCESSED_DATA_DIR / "sms_final.csv",
    index=False
)

phishing_final.to_parquet(
    PROCESSED_DATA_DIR / "phishing_final.parquet",
    index=False
)

print("Modeling datasets saved successfully.")

Modeling datasets saved successfully.


## Train, Validation and Test Splits
I divide both datasets into training, validation, and test partitions using stratified sampling. This preserves the original class distributions while preventing exact and normalized duplicate messages from appearing across partitions.

I use a fixed random seed to make the splits reproducible. The test sets will remain untouched during model selection and hyperparameter tuning.

Because the phishing dataset contains unreliable timestamps, I use stratified random splitting rather than chronological splitting. I will also evaluate potential cross-split similarity before interpreting the final model results.

In [59]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42


def create_splits(df, label_column="label"):
    """
    Create reproducible 70/15/15 stratified splits.
    """
    train_df, temp_df = train_test_split(
        df,
        test_size=0.30,
        stratify=df[label_column],
        random_state=RANDOM_STATE
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        stratify=temp_df[label_column],
        random_state=RANDOM_STATE
    )

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True)
    )

In [61]:
# I create separate partitions for spam and phishing classification.
sms_train, sms_val, sms_test = create_splits(sms_final)

phishing_train, phishing_val, phishing_test = create_splits(
    phishing_final
)

# I summarize the resulting partitions.
for name, splits in {
    "SMS": (sms_train, sms_val, sms_test),
    "PHISHING": (phishing_train, phishing_val, phishing_test)
}.items():

    print(f"\n{name} DATASET")

    for split_name, split_df in zip(
        ["Train", "Validation", "Test"],
        splits
    ):
        print(
            f"{split_name}: "
            f"{len(split_df):,} records | "
            f"Positive class: "
            f"{split_df['label'].mean():.2%}"
        )


SMS DATASET
Train: 3,591 records | Positive class: 12.31%
Validation: 769 records | Positive class: 12.22%
Test: 770 records | Positive class: 12.34%

PHISHING DATASET
Train: 73,045 records | Positive class: 44.57%
Validation: 15,653 records | Positive class: 44.57%
Test: 15,653 records | Positive class: 44.57%


In [63]:
# I preserve the exact partitions for reproducible experiments.
split_dir = PROCESSED_DATA_DIR / "splits"
split_dir.mkdir(parents=True, exist_ok=True)

for name, partitions in {
    "sms": (sms_train, sms_val, sms_test),
    "phishing": (phishing_train, phishing_val, phishing_test)
}.items():

    for split_name, partition in zip(
        ["train", "val", "test"],
        partitions
    ):
        output_path = split_dir / f"{name}_{split_name}.parquet"

        partition.to_parquet(
            output_path,
            index=False
        )

print("All six dataset partitions saved successfully.")

All six dataset partitions saved successfully.


## Cross-Split Near-Duplicate Assessment
I examine similarity between messages in different dataset partitions to identify potential leakage from repeated spam or phishing templates.

I use character n-gram TF-IDF features and sparse similarity comparisons rather than computing a dense pairwise similarity matrix. I treat high similarity as a warning for further investigation, not automatic proof that two messages are duplicates.

This audit is separate from model training and will help me determine whether the current splits provide a credible evaluation.

In [67]:
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# I use character n-grams to identify messages with
# similar wording despite minor formatting differences.
similarity_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=1,
    sublinear_tf=True,
    dtype=np.float32
)

# I fit the similarity representation on training SMS messages.
X_sms_train = similarity_vectorizer.fit_transform(
    sms_train["text"]
)

X_sms_val = similarity_vectorizer.transform(
    sms_val["text"]
)

X_sms_test = similarity_vectorizer.transform(
    sms_test["text"]
)

print("Training matrix:", X_sms_train.shape)
print("Validation matrix:", X_sms_val.shape)
print("Test matrix:", X_sms_test.shape)

Training matrix: (3591, 121782)
Validation matrix: (769, 121782)
Test matrix: (770, 121782)


In [69]:
# I compare validation and test messages with training messages.
# Cosine distance is suitable for normalized TF-IDF vectors.
nn_model = NearestNeighbors(
    n_neighbors=1,
    metric="cosine",
    algorithm="brute",
    n_jobs=1
)

nn_model.fit(X_sms_train)

val_distances, val_indices = nn_model.kneighbors(X_sms_val)
test_distances, test_indices = nn_model.kneighbors(X_sms_test)

# I convert cosine distance to cosine similarity.
val_similarity = 1 - val_distances.ravel()
test_similarity = 1 - test_distances.ravel()

print("SMS CROSS-SPLIT SIMILARITY")

for name, scores in [
    ("Validation", val_similarity),
    ("Test", test_similarity)
]:
    print(f"\n{name}")
    print("Mean similarity:", round(float(scores.mean()), 4))
    print("95th percentile:", round(float(np.percentile(scores, 95)), 4))
    print("Similarity >= 0.90:", int((scores >= 0.90).sum()))
    print("Similarity >= 0.95:", int((scores >= 0.95).sum()))

SMS CROSS-SPLIT SIMILARITY

Validation
Mean similarity: 0.2825
95th percentile: 0.7107
Similarity >= 0.90: 14
Similarity >= 0.95: 6

Test
Mean similarity: 0.29
95th percentile: 0.7497
Similarity >= 0.90: 15
Similarity >= 0.95: 6


## Inspect High-Similarity SMS Pairs
I examine the most similar validation and test messages alongside their nearest training examples. This helps me distinguish genuine near-duplicate templates from messages that share common wording.

I use the inspection results to evaluate potential leakage before training any classification models.

In [73]:
import pandas as pd
import numpy as np


def inspect_similar_pairs(
    query_df,
    reference_df,
    similarities,
    nearest_indices,
    top_n=10
):
    """
    Compare evaluation messages with their nearest
    training examples.
    """

    top_indices = np.argsort(similarities)[::-1][:top_n]

    records = []

    for query_idx in top_indices:
        train_idx = int(nearest_indices[query_idx, 0])

        records.append({
            "similarity": round(float(similarities[query_idx]), 4),
            "evaluation_label": int(
                query_df.iloc[query_idx]["label"]
            ),
            "training_label": int(
                reference_df.iloc[train_idx]["label"]
            ),
            "evaluation_text": query_df.iloc[query_idx]["text"],
            "training_text": reference_df.iloc[train_idx]["text"]
        })

    return pd.DataFrame(records)

In [75]:
print("TOP VALIDATION MATCHES")

validation_pairs = inspect_similar_pairs(
    sms_val,
    sms_train,
    val_similarity,
    val_indices,
    top_n=10
)

display(validation_pairs)


print("\nTOP TEST MATCHES")

test_pairs = inspect_similar_pairs(
    sms_test,
    sms_train,
    test_similarity,
    test_indices,
    top_n=10
)

display(test_pairs)

TOP VALIDATION MATCHES


,similarity,evaluation_label,training_label,evaluation_text,training_text
0,0.9917,1,1,U have a secret admirer who is looking 2 make ...,U have a secret admirer who is looking 2 make ...
1,0.9769,1,1,Moby Pub Quiz.Win a £100 High Street prize if ...,Moby Pub Quiz.Win a £100 High Street prize if ...
2,0.9636,1,1,"As a valued customer, I am pleased to advise y...","As a valued customer, I am pleased to advise y..."
3,0.9609,1,1,URGENT! Your Mobile number has been awarded wi...,URGENT! Your Mobile number has been awarded wi...
4,0.9588,1,1,Congratulations! Thanks to a good friend U hav...,Congratulations! Thanks to a good friend U hav...
5,0.9550,1,1,December only! Had your mobile 11mths+? You ar...,December only! Had your mobile 11mths+? You ar...
6,0.9466,1,1,New TEXTBUDDY Chat 2 horny guys in ur area 4 j...,New TEXTBUDDY Chat 2 horny guys in ur area 4 j...
7,0.9451,1,1,4mths half price Orange line rental & latest c...,4mths half price Orange line rental & latest c...
8,0.9445,1,1,8007 FREE for 1st week! No1 Nokia tone 4 ur mo...,FREE for 1st week! No1 Nokia tone 4 ur mob eve...
9,0.9435,1,1,"URGENT, IMPORTANT INFORMATION FOR O2 USER. TOD...",Message Important information for O2 user. Tod...



TOP TEST MATCHES


,similarity,evaluation_label,training_label,evaluation_text,training_text
0,0.9984,1,1,ree entry in 2 a weekly comp for a chance to w...,Free entry in 2 a weekly comp for a chance to ...
1,0.9945,1,1,"ou are guaranteed the latest Nokia Phone, a 40...","You are guaranteed the latest Nokia Phone, a 4..."
2,0.9690,0,0,Beautiful Truth against Gravity.. Read careful...,Beautiful Truth against Gravity.. Read careful...
3,0.9652,0,0,"Night has ended for another day, morning has c...","Night has ended for another day, morning has c..."
4,0.9615,1,1,URGENT! We are trying to contact U. Todays dra...,URGENT! We are trying to contact U. Todays dra...
5,0.9527,1,1,HOT LIVE FANTASIES call now 08707509020 Just 2...,HOT LIVE FANTASIES call now 08707500020 Just 2...
6,0.9442,1,1,Ur cash-balance is currently 500 pounds - to m...,Ur cash-balance is currently 500 pounds - to m...
7,0.9403,0,0,Living is very simple.. Loving is also simple....,Living is very simple.. Loving is also simple....
8,0.9391,1,1,U have a Secret Admirer who is looking 2 make ...,U have a Secret Admirer who is looking 2 make ...
9,0.9382,1,1,You have won a guaranteed £200 award or even £...,You have won a guaranteed £200 award or even £...


## Create Template-Aware Groups
I identified near-duplicate SMS messages across my initial training and evaluation partitions. Several examples are variations of the same spam campaigns.

To reduce evaluation leakage, I group messages with high character n-gram similarity before creating new partitions. This ensures closely related message templates remain together.

I use a similarity threshold of 0.90 as a practical starting point and retain every message rather than removing valid campaign variations.

In [78]:
import numpy as np

from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# I construct character n-gram features for grouping.
group_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=1,
    sublinear_tf=True,
    dtype=np.float32
)

X_group = group_vectorizer.fit_transform(
    sms_final["text"]
)

# I identify neighbors with cosine similarity >= 0.90.
neighbors = NearestNeighbors(
    radius=0.10,
    metric="cosine",
    algorithm="brute",
    n_jobs=1
)

neighbors.fit(X_group)

# I construct a sparse graph connecting similar messages.
similarity_graph = neighbors.radius_neighbors_graph(
    X_group,
    mode="connectivity"
)

# I identify connected components.
n_groups, group_ids = connected_components(
    similarity_graph,
    directed=False
)

sms_grouped = sms_final.copy()
sms_grouped["group_id"] = group_ids

print("Total messages:", len(sms_grouped))
print("Template groups:", n_groups)
print("Groups containing multiple messages:",
      (sms_grouped.groupby("group_id").size() > 1).sum())

print("Largest template group:",
      sms_grouped.groupby("group_id").size().max())

Total messages: 5130
Template groups: 5069
Groups containing multiple messages: 56
Largest template group: 3


## Create Group-Aware SMS Splits
I create new SMS training, validation, and test partitions using the template groups identified during similarity analysis.

My objective is to prevent closely related messages from appearing across partitions while preserving the original class distribution as closely as possible.

I will verify group separation, partition sizes, and class balance before replacing the earlier random splits.

In [81]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np

RANDOM_STATE = 42

# I use seven folds to approximate a 70/15/15 split.
# Five folds will form training, one validation, and one testing.
sgkf = StratifiedGroupKFold(
    n_splits=7,
    shuffle=True,
    random_state=RANDOM_STATE
)

X = sms_grouped["text"]
y = sms_grouped["label"]
groups = sms_grouped["group_id"]

folds = list(
    sgkf.split(X, y, groups)
)

# I collect the validation indices from each fold.
fold_indices = [test_idx for _, test_idx in folds]

# I assign five folds to training and one each to validation/test.
train_indices = np.concatenate(fold_indices[:5])
val_indices = fold_indices[5]
test_indices = fold_indices[6]

# I create the new partitions.
sms_train_grouped = (
    sms_grouped.iloc[train_indices]
    .drop(columns="group_id")
    .reset_index(drop=True)
)

sms_val_grouped = (
    sms_grouped.iloc[val_indices]
    .drop(columns="group_id")
    .reset_index(drop=True)
)

sms_test_grouped = (
    sms_grouped.iloc[test_indices]
    .drop(columns="group_id")
    .reset_index(drop=True)
)

print("GROUP-AWARE SMS SPLITS")

for name, df in [
    ("Train", sms_train_grouped),
    ("Validation", sms_val_grouped),
    ("Test", sms_test_grouped)
]:
    print(
        f"{name}: {len(df):,} messages | "
        f"Spam: {df['label'].mean():.2%}"
    )

GROUP-AWARE SMS SPLITS
Train: 3,665 messages | Spam: 12.31%
Validation: 733 messages | Spam: 12.28%
Test: 732 messages | Spam: 12.30%


In [83]:
# I verify that related templates never cross partitions.
train_groups = set(groups.iloc[train_indices])
val_groups = set(groups.iloc[val_indices])
test_groups = set(groups.iloc[test_indices])

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

# I ensure every record appears in exactly one partition.
all_indices = np.concatenate([
    train_indices,
    val_indices,
    test_indices
])

assert len(all_indices) == len(sms_grouped)
assert len(np.unique(all_indices)) == len(sms_grouped)

print("Group separation: PASSED")
print("Record coverage: PASSED")

Group separation: PASSED
Record coverage: PASSED


## Saving the Final SMS Splits
I replaced my initial random SMS split with a template-aware split after discovering near-duplicate messages across the original partitions.

The revised partitions preserve the spam class distribution while keeping identified message templates together. I verified that no template group crosses partitions and that every message appears exactly once.

I will use these partitions consistently throughout model training, validation, and final evaluation.

In [87]:
# I save the validated group-aware SMS partitions.
sms_partitions = {
    "train": sms_train_grouped,
    "val": sms_val_grouped,
    "test": sms_test_grouped,
}

for split_name, partition in sms_partitions.items():
    output_path = split_dir / f"sms_{split_name}.parquet"

    partition.to_parquet(
        output_path,
        index=False
    )

    print(f"Saved: {output_path.name} ({len(partition):,} records)")

Saved: sms_train.parquet (3,665 records)
Saved: sms_val.parquet (733 records)
Saved: sms_test.parquet (732 records)


In [89]:
# I verify that the saved partitions retain their expected sizes.
for split_name, expected_df in sms_partitions.items():
    saved_path = split_dir / f"sms_{split_name}.parquet"
    reloaded_df = pd.read_parquet(saved_path)

    pd.testing.assert_frame_equal(
        reloaded_df,
        expected_df
    )

    print(f"{split_name.upper()}: VERIFIED")

TRAIN: VERIFIED
VAL: VERIFIED
TEST: VERIFIED


## Assess Repeated Templates in Phishing Emails
I examine repeated email templates before creating the final phishing train, validation, and test partitions.

Because this dataset is substantially larger than the SMS dataset, I use MinHash Locality-Sensitive Hashing to identify candidate near-duplicate messages efficiently.

I will verify candidate matches before grouping emails, preserving their labels and avoiding unnecessary removal of valid records.

In [93]:
# I use the existing modeling-ready phishing dataset.
phishing_grouped = phishing_final.copy()

def normalize_email_template(text: str) -> str:
    """
    Normalize email text for template similarity analysis.
    This representation is not used for model training.
    """
    text = str(text).lower()

    # I normalize whitespace and repeated formatting.
    text = re.sub(r"\s+", " ", text)

    return text.strip()


phishing_grouped["template_text"] = (
    phishing_grouped["text"]
    .map(normalize_email_template)
)

# I inspect text lengths before building MinHash signatures.
phishing_grouped["text_length"] = (
    phishing_grouped["template_text"].str.len()
)

print("PHISHING TEMPLATE PREPARATION")

print("Total emails:", len(phishing_grouped))

print("\nText length distribution:")
print(
    phishing_grouped["text_length"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nVery short emails (< 30 characters):")
print(
    (phishing_grouped["text_length"] < 30).sum()
)

PHISHING TEMPLATE PREPARATION
Total emails: 104351

Text length distribution:
count    104351.000000
mean       1209.867974
std        1129.890175
min           3.000000
25%         429.000000
50%         842.000000
75%        1572.000000
90%        2862.000000
95%        3641.000000
99%        4913.500000
max       30892.000000
Name: text_length, dtype: float64

Very short emails (< 30 characters):
190


## The MinHash Signature Function
I use word-level shingles and MinHash signatures to identify potentially repeated phishing email templates.

MinHash approximates Jaccard similarity between sets of text shingles. I use it to generate candidate matches efficiently before verifying similarity.

I initially test the implementation on a small sample to check its behavior before processing the full dataset.

In [96]:
from datasketch import MinHash

NUM_PERM = 64
SHINGLE_SIZE = 5


def create_word_shingles(text: str, size: int = SHINGLE_SIZE) -> set[str]:
    """
    Convert text into a set of consecutive word shingles.
    """
    words = re.findall(r"\b\w+\b", text.lower())

    if len(words) < size:
        return set()

    return {
        " ".join(words[i:i + size])
        for i in range(len(words) - size + 1)
    }


def create_minhash(shingles: set[str]) -> MinHash:
    """
    Generate a deterministic MinHash signature.
    """
    signature = MinHash(num_perm=NUM_PERM)

    for shingle in shingles:
        signature.update(shingle.encode("utf-8"))

    return signature

In [98]:
# I test whether similar messages share substantial shingle overlap.
message_a = (
    "Your account has been temporarily suspended. "
    "Please verify your account details immediately."
)

message_b = (
    "Your account has been temporarily suspended. "
    "Please verify your account details today."
)

message_c = (
    "The project meeting has been moved to Friday afternoon."
)

messages = [message_a, message_b, message_c]

shingle_sets = [
    create_word_shingles(message)
    for message in messages
]

signatures = [
    create_minhash(shingles)
    for shingles in shingle_sets
]

for i, j in [(0, 1), (0, 2)]:
    intersection = len(shingle_sets[i] & shingle_sets[j])
    union = len(shingle_sets[i] | shingle_sets[j])

    exact_jaccard = intersection / union if union else 0.0

    estimated_jaccard = signatures[i].jaccard(signatures[j])

    print(f"Message {i + 1} vs Message {j + 1}")
    print(f"Exact Jaccard: {exact_jaccard:.4f}")
    print(f"MinHash estimate: {estimated_jaccard:.4f}")
    print()

Message 1 vs Message 2
Exact Jaccard: 0.7778
MinHash estimate: 0.7188

Message 1 vs Message 3
Exact Jaccard: 0.0000
MinHash estimate: 0.0000



## Test MinHash LSH on a Small Sample
I test MinHash Locality-Sensitive Hashing on a reproducible sample of 2,000 emails before applying it to the complete phishing dataset.

I use LSH to retrieve potentially similar messages and calculate exact Jaccard similarity for the retrieved candidates. This allows me to assess candidate quality and computational cost.

I treat this as an engineering feasibility test rather than a complete evaluation of LSH recall.

In [101]:
import time

from datasketch import MinHashLSH

# I configure the pilot experiment.
PILOT_SIZE = 2_000
LSH_THRESHOLD = 0.85

# I use a fixed seed to make the sample reproducible.
pilot_df = phishing_grouped.sample(
    n=PILOT_SIZE,
    random_state=42
).reset_index(drop=True)

# I initialize the LSH index.
lsh = MinHashLSH(
    threshold=LSH_THRESHOLD,
    num_perm=NUM_PERM
)

# I cache shingles for exact similarity verification.
shingle_cache = {}

candidate_pairs = set()
verified_pairs = []

start_time = time.perf_counter()

for idx, text in enumerate(pilot_df["template_text"]):

    # I generate the word shingles.
    shingles = create_word_shingles(text)

    # I skip messages that cannot form five-word shingles.
    # These require a separate short-message strategy.
    if not shingles:
        continue

    signature = create_minhash(shingles)

    # I search for candidates already added to the index.
    candidates = lsh.query(signature)

    for candidate in candidates:

        pair = (int(candidate), idx)

        if pair in candidate_pairs:
            continue

        candidate_pairs.add(pair)

        other_shingles = shingle_cache[int(candidate)]

        intersection = len(shingles & other_shingles)
        union = len(shingles | other_shingles)

        exact_similarity = (
            intersection / union if union else 0.0
        )

        if exact_similarity >= LSH_THRESHOLD:
            verified_pairs.append(
                (int(candidate), idx, exact_similarity)
            )

    # I insert the current message after querying.
    lsh.insert(str(idx), signature)

    # I retain its shingles for candidate verification.
    shingle_cache[idx] = shingles

elapsed = time.perf_counter() - start_time

print("MINHASH LSH PILOT RESULTS")
print("Sample size:", len(pilot_df))
print("Indexed messages:", len(shingle_cache))
print("Candidate pairs:", len(candidate_pairs))
print("Verified pairs:", len(verified_pairs))
print(f"Elapsed time: {elapsed:.2f} seconds")

MINHASH LSH PILOT RESULTS
Sample size: 2000
Indexed messages: 1996
Candidate pairs: 6
Verified pairs: 4
Elapsed time: 10.16 seconds


In [103]:
# I summarize verified near-duplicate similarities
# without displaying potentially sensitive email content.

if verified_pairs:

    verified_scores = [
        pair[2] for pair in verified_pairs
    ]

    print("VERIFIED SIMILARITY STATISTICS")
    print("Minimum:", round(min(verified_scores), 4))
    print("Mean:", round(float(np.mean(verified_scores)), 4))
    print("Maximum:", round(max(verified_scores), 4))

    print("\nSample verified pairs:")
    print(
        pd.DataFrame(
            verified_pairs[:10],
            columns=[
                "email_a",
                "email_b",
                "jaccard_similarity"
            ]
        )
    )

else:
    print("No verified near-duplicate pairs found.")

VERIFIED SIMILARITY STATISTICS
Minimum: 0.8851
Mean: 0.9415
Maximum: 0.9699

Sample verified pairs:
   email_a  email_b  jaccard_similarity
0      142     1013            0.958333
1     1013     1421            0.969880
2      142     1421            0.952663
3     1264     1961            0.885057


## Prepare Full-Corpus Template Grouping
I extend the MinHash LSH approach to the complete phishing dataset after validating its behavior on a smaller sample.

I use LSH to retrieve candidate near-duplicate emails and verify candidate pairs using exact Jaccard similarity. Verified pairs will form connected template groups, allowing related messages to remain together during dataset splitting.

I avoid storing all shingle sets in memory and track processing progress to monitor computational feasibility.

In [106]:
# I configure full-corpus template grouping.
LSH_THRESHOLD = 0.85
NUM_PERM = 64
SHINGLE_SIZE = 5
BATCH_SIZE = 5_000

n_emails = len(phishing_grouped)

# I initialize the LSH index.
full_lsh = MinHashLSH(
    threshold=LSH_THRESHOLD,
    num_perm=NUM_PERM
)

# I initialize the disjoint-set structure used
# to form connected components of similar emails.
parent = np.arange(n_emails)
rank = np.zeros(n_emails, dtype=np.int8)


def find_root(index):
    """Find the representative of a template group."""
    index = int(index)

    while parent[index] != index:
        parent[index] = parent[parent[index]]
        index = int(parent[index])

    return index


def union_groups(index_a, index_b):
    """Merge two verified near-duplicate groups."""
    root_a = find_root(index_a)
    root_b = find_root(index_b)

    if root_a == root_b:
        return

    if rank[root_a] < rank[root_b]:
        parent[root_a] = root_b

    elif rank[root_a] > rank[root_b]:
        parent[root_b] = root_a

    else:
        parent[root_b] = root_a
        rank[root_a] += 1


print("FULL-CORPUS GROUPING CONFIGURATION")
print("Emails:", n_emails)
print("LSH threshold:", LSH_THRESHOLD)
print("MinHash permutations:", NUM_PERM)
print("Shingle size:", SHINGLE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Grouping structures initialized successfully.")

FULL-CORPUS GROUPING CONFIGURATION
Emails: 104351
LSH threshold: 0.85
MinHash permutations: 64
Shingle size: 5
Batch size: 5000
Grouping structures initialized successfully.


## Full-Corpus Phishing Template Grouping
I identify near-duplicate templates across the complete phishing email dataset using MinHash LSH.

For each email, I generate a MinHash signature, retrieve previously indexed candidate messages, and verify their similarity using exact Jaccard similarity.

I merge verified near-duplicate messages into connected groups. These groups will later help me create train, validation, and test partitions that reduce leakage from repeated email templates.

I process the emails sequentially, monitor progress, and avoid storing every email's shingle set in memory.

In [109]:
# I track the number of candidate comparisons and verified matches.
candidate_count = 0
verified_count = 0
short_message_count = 0

start_time = time.perf_counter()

# I use positional access so each email has a stable integer index.
email_texts = phishing_grouped["template_text"].tolist()

for idx, text in enumerate(email_texts):

    # I construct word shingles for the current email.
    shingles = create_word_shingles(text)

    # I handle messages without enough words separately.
    # They remain independent groups for now.
    if not shingles:
        short_message_count += 1
        continue

    # I generate the current email's MinHash signature.
    signature = create_minhash(shingles)

    # I retrieve potential matches from previously indexed emails.
    candidates = full_lsh.query(signature)

    for candidate in candidates:

        other_idx = int(candidate)
        candidate_count += 1

        # I reconstruct the candidate's shingles only when necessary.
        other_shingles = create_word_shingles(
            email_texts[other_idx]
        )

        # I calculate exact Jaccard similarity.
        intersection = len(shingles & other_shingles)
        union_size = len(shingles | other_shingles)

        similarity = (
            intersection / union_size
            if union_size else 0.0
        )

        # I connect sufficiently similar emails.
        if similarity >= LSH_THRESHOLD:
            union_groups(idx, other_idx)
            verified_count += 1

    # I add the current email to the LSH index
    # after checking previously indexed messages.
    full_lsh.insert(str(idx), signature)

    # I display progress after each batch.
    if (idx + 1) % BATCH_SIZE == 0:
        elapsed = time.perf_counter() - start_time

        print(
            f"Processed {idx + 1:,}/{n_emails:,} emails | "
            f"Candidates: {candidate_count:,} | "
            f"Verified: {verified_count:,} | "
            f"Elapsed: {elapsed:.1f}s",
            flush=True
        )

elapsed = time.perf_counter() - start_time

print("\nFULL-CORPUS GROUPING COMPLETED")
print("Total emails:", n_emails)
print("Indexed emails:", n_emails - short_message_count)
print("Short messages skipped:", short_message_count)
print("Candidate comparisons:", candidate_count)
print("Verified pairs:", verified_count)
print(f"Total processing time: {elapsed:.2f} seconds")

Processed 5,000/104,351 emails | Candidates: 141 | Verified: 101 | Elapsed: 19.2s
Processed 10,000/104,351 emails | Candidates: 275 | Verified: 195 | Elapsed: 39.6s
Processed 15,000/104,351 emails | Candidates: 798 | Verified: 595 | Elapsed: 61.5s
Processed 20,000/104,351 emails | Candidates: 1,114 | Verified: 821 | Elapsed: 83.0s
Processed 25,000/104,351 emails | Candidates: 1,424 | Verified: 1,019 | Elapsed: 104.4s
Processed 30,000/104,351 emails | Candidates: 1,931 | Verified: 1,396 | Elapsed: 125.8s
Processed 35,000/104,351 emails | Candidates: 3,368 | Verified: 2,081 | Elapsed: 149.0s
Processed 40,000/104,351 emails | Candidates: 3,762 | Verified: 2,375 | Elapsed: 172.7s
Processed 45,000/104,351 emails | Candidates: 7,218 | Verified: 4,709 | Elapsed: 194.3s
Processed 50,000/104,351 emails | Candidates: 8,185 | Verified: 5,238 | Elapsed: 214.7s
Processed 55,000/104,351 emails | Candidates: 9,029 | Verified: 5,882 | Elapsed: 235.8s
Processed 60,000/104,351 emails | Candidates: 10,04

In [111]:
# I assign each email to its connected template group.
phishing_grouped["group_id"] = [
    find_root(idx)
    for idx in range(n_emails)
]

group_sizes = phishing_grouped.groupby("group_id").size()

print("PHISHING TEMPLATE GROUP SUMMARY")
print("Total emails:", len(phishing_grouped))
print("Unique template groups:", len(group_sizes))
print(
    "Groups containing multiple emails:",
    int((group_sizes > 1).sum())
)
print("Largest template group:", int(group_sizes.max()))

print("\nGROUP SIZE DISTRIBUTION")
print(
    group_sizes.value_counts()
    .sort_index()
    .head(15)
)

PHISHING TEMPLATE GROUP SUMMARY
Total emails: 104351
Unique template groups: 100838
Groups containing multiple emails: 1623
Largest template group: 83

GROUP SIZE DISTRIBUTION
1     99215
2      1182
3       224
4        75
5        31
6        22
7        19
8         7
9         6
10        3
11        5
12        6
13        5
14        6
15        3
Name: count, dtype: int64


## Validate Group Labels Before Splitting
I validate the phishing template groups before creating the final dataset partitions.

I examine whether any groups contain conflicting classifications and whether large groups are concentrated in particular source datasets.

This helps me assess whether the grouping method is appropriate for leakage-aware splitting without modifying the original labels.

In [114]:
# I summarize label consistency within each template group.
group_label_counts = (
    phishing_grouped
    .groupby("group_id")["label"]
    .nunique()
)

conflicting_group_ids = group_label_counts[
    group_label_counts > 1
].index

print("PHISHING GROUP VALIDATION")
print("Total groups:", len(group_label_counts))
print("Groups with conflicting labels:", len(conflicting_group_ids))

# I count records belonging to conflicting groups.
conflicting_records = phishing_grouped[
    phishing_grouped["group_id"].isin(conflicting_group_ids)
]

print("Records in conflicting groups:", len(conflicting_records))


# I examine the largest groups without displaying email content.
group_summary = (
    phishing_grouped
    .groupby("group_id")
    .agg(
        records=("label", "size"),
        unique_labels=("label", "nunique"),
        phishing_rate=("label", "mean"),
        unique_sources=("source", "nunique")
    )
    .sort_values("records", ascending=False)
)

print("\nTOP 10 LARGEST GROUPS")
display(group_summary.head(10))

PHISHING GROUP VALIDATION
Total groups: 100838
Groups with conflicting labels: 3
Records in conflicting groups: 6

TOP 10 LARGEST GROUPS


,records,unique_labels,phishing_rate,unique_sources
group_id,,,,
39905,83,1,1.0,1
95111,79,1,1.0,1
66763,56,1,1.0,1
64062,44,1,0.0,1
48080,43,1,1.0,1
63459,39,1,0.0,1
62757,37,1,0.0,1
79980,37,1,1.0,1
61838,37,1,1.0,1


## Create Group-Aware Phishing Splits
I create group-aware training, validation, and test partitions for the phishing dataset.

I keep related email templates within the same partition to reduce evaluation leakage. I preserve all original labels, including the small number of groups containing both classes.

I use a reproducible allocation procedure that targets a 70/15/15 split while maintaining approximately the same class distribution across partitions.

In [117]:
RANDOM_STATE = 42

# I summarize the records and labels within each template group.
group_info = (
    phishing_grouped
    .groupby("group_id")
    .agg(
        group_size=("label", "size"),
        positive_count=("label", "sum")
    )
    .reset_index()
)

group_info["negative_count"] = (
    group_info["group_size"] - group_info["positive_count"]
)

# I assign a majority-class label for stratified allocation.
# This does not modify any individual message labels.
group_info["stratum"] = (
    group_info["positive_count"] >=
    group_info["negative_count"]
).astype("int8")

print("GROUP-LEVEL SPLITTING METADATA")
print("Groups:", len(group_info))
print("Total records:", group_info["group_size"].sum())

print("\nGroup strata:")
print(group_info["stratum"].value_counts())

print("\nGroup size statistics:")
print(group_info["group_size"].describe())

GROUP-LEVEL SPLITTING METADATA
Groups: 100838
Total records: 104351

Group strata:
stratum
0    56495
1    44343
Name: count, dtype: int64

Group size statistics:
count    100838.000000
mean          1.034838
std           0.662909
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          83.000000
Name: group_size, dtype: float64


## Create the Final Phishing Splits
I create reproducible training, validation, and test partitions by splitting the identified email template groups.

I stratify the groups using their majority class while retaining every email's original label. This prevents identified near-duplicate templates from crossing partitions.

I target a 70/15/15 allocation and verify the resulting record counts, class distributions, and group separation before saving the partitions.

In [120]:
RANDOM_STATE = 42

# I separate 70% of the template groups for training.
train_groups_df, temp_groups_df = train_test_split(
    group_info,
    test_size=0.30,
    stratify=group_info["stratum"],
    random_state=RANDOM_STATE
)

# I divide the remaining groups equally between
# validation and testing.
val_groups_df, test_groups_df = train_test_split(
    temp_groups_df,
    test_size=0.50,
    stratify=temp_groups_df["stratum"],
    random_state=RANDOM_STATE
)

# I collect the group IDs belonging to each partition.
train_group_ids = set(train_groups_df["group_id"])
val_group_ids = set(val_groups_df["group_id"])
test_group_ids = set(test_groups_df["group_id"])

print("GROUP ALLOCATION")
print("Train groups:", len(train_group_ids))
print("Validation groups:", len(val_group_ids))
print("Test groups:", len(test_group_ids))

GROUP ALLOCATION
Train groups: 70586
Validation groups: 15126
Test groups: 15126


In [123]:
# I assign each email according to its template group.
phishing_train_grouped = phishing_grouped[
    phishing_grouped["group_id"].isin(train_group_ids)
].copy()

phishing_val_grouped = phishing_grouped[
    phishing_grouped["group_id"].isin(val_group_ids)
].copy()

phishing_test_grouped = phishing_grouped[
    phishing_grouped["group_id"].isin(test_group_ids)
].copy()

# I retain only the fields needed for modeling
# and subsequent evaluation.
model_columns = [
    "subject",
    "body",
    "text",
    "label",
    "source",
    "date"
]

phishing_partitions = {
    "train": phishing_train_grouped[model_columns].reset_index(drop=True),
    "val": phishing_val_grouped[model_columns].reset_index(drop=True),
    "test": phishing_test_grouped[model_columns].reset_index(drop=True)
}

print("FINAL PHISHING SPLITS")

for split_name, partition in phishing_partitions.items():
    print(
        f"{split_name.upper()}: "
        f"{len(partition):,} records | "
        f"Phishing: {partition['label'].mean():.2%}"
    )

FINAL PHISHING SPLITS
TRAIN: 73,128 records | Phishing: 44.60%
VAL: 15,606 records | Phishing: 44.44%
TEST: 15,617 records | Phishing: 44.54%


In [125]:
# I verify that no template group belongs to multiple partitions.
assert train_group_ids.isdisjoint(val_group_ids)
assert train_group_ids.isdisjoint(test_group_ids)
assert val_group_ids.isdisjoint(test_group_ids)

print("Group separation: PASSED")

Group separation: PASSED


In [127]:
# I verify that every email was assigned exactly once.
total_records = sum(
    len(partition)
    for partition in phishing_partitions.values()
)

assert total_records == len(phishing_final)

# I verify that all original group IDs were assigned.
all_group_ids = (
    train_group_ids |
    val_group_ids |
    test_group_ids
)

assert all_group_ids == set(group_info["group_id"])

print("Record coverage: PASSED")

Record coverage: PASSED


In [129]:
# I compare class distributions across partitions.
split_summary = pd.DataFrame([
    {
        "split": name,
        "records": len(df),
        "percentage": round(
            len(df) / len(phishing_final) * 100, 2
        ),
        "phishing_rate": round(
            df["label"].mean() * 100, 2
        )
    }
    for name, df in phishing_partitions.items()
])

display(split_summary)

,split,records,percentage,phishing_rate
0,train,73128,70.08,44.60
1,val,15606,14.96,44.44
2,test,15617,14.97,44.54


## Save and Verify the Final Phishing Splits
I finalized my phishing dataset partitions using template-aware grouping to reduce leakage from repeated email campaigns.

I assigned related messages to the same partition and verified that the resulting splits preserve the original class distribution. I also confirmed that every email appears in exactly one partition.

I will use these fixed partitions for model development, keeping the test set untouched until final evaluation.

In [132]:
# I save the validated phishing dataset partitions.
for split_name, partition in phishing_partitions.items():
    output_path = split_dir / f"phishing_{split_name}.parquet"

    partition.to_parquet(
        output_path,
        index=False
    )

    print(f"Saved: {output_path.name} ({len(partition):,} records)")

Saved: phishing_train.parquet (73,128 records)
Saved: phishing_val.parquet (15,606 records)
Saved: phishing_test.parquet (15,617 records)


In [134]:
# I reload each partition to confirm that its contents were preserved.
for split_name, expected_df in phishing_partitions.items():
    saved_path = split_dir / f"phishing_{split_name}.parquet"

    reloaded_df = pd.read_parquet(saved_path)

    pd.testing.assert_frame_equal(
        reloaded_df,
        expected_df
    )

    print(f"{split_name.upper()}: VERIFIED")

TRAIN: VERIFIED
VAL: VERIFIED
TEST: VERIFIED


In [136]:
import json

# I record the dataset preparation settings for reproducibility.
split_metadata = {
    "random_state": 42,
    "sms": {
        "records": 5130,
        "train": 3665,
        "validation": 733,
        "test": 732,
        "grouping_method": "character_ngram_cosine_connected_components",
        "similarity_threshold": 0.90
    },
    "phishing": {
        "records": 104351,
        "train": 73128,
        "validation": 15606,
        "test": 15617,
        "grouping_method": "minhash_lsh_verified_jaccard",
        "jaccard_threshold": 0.85,
        "num_perm": 64,
        "word_shingle_size": 5,
        "skipped_short_messages": 191
    },
    "limitations": [
        "Phishing timestamps contain implausible values and were not used for chronological splitting.",
        "Approximate similarity grouping does not guarantee removal of all near-duplicate templates.",
        "Group-aware splitting reduces template leakage but does not establish performance on future phishing campaigns."
    ]
}

metadata_path = PROCESSED_DATA_DIR / "split_metadata.json"

with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(split_metadata, file, indent=2)

print("Split metadata saved:", metadata_path.name)

Split metadata saved: split_metadata.json
